# Airlines Delay Data

## Data Acquisition and Pre-Processing

In [ ]:
import pandas as pdimport numpy as npimport seaborn as snsfrom matplotlib import pyplot as pltfrom sklearn.model_selection import train_test_splitfrom sklearn.datasets import make_classification

In [ ]:
df = pd.read_csv("datasets/airlines_delay.csv", sep=",")df.info()# label encode categorical variablesAirlineUnique = df.Airline.unique()AirportFromUnique = df.AirportFrom.unique()AirportToUnique = df.AirportTo.unique()Airlinelst = list(range(len(AirlineUnique)))df['NumAirline'] = df['Airline']df['NumAirline'].replace(AirlineUnique, Airlinelst, inplace=True)AirportFromlst = list(range(len(AirportFromUnique)))df['NumAirportFrom'] = df['AirportFrom']df['NumAirportFrom'].replace(AirportFromUnique, AirportFromlst, inplace=True)AirportTolst = list(range(len(AirportToUnique)))df['NumAirportTo'] = df['AirportTo']df['NumAirportTo'].replace(AirportToUnique, AirportTolst, inplace=True)# reduce dataset sizedf = df.sample(n=10000)X = df[['Length','NumAirline','NumAirportFrom','NumAirportTo','DayOfWeek']]y = df['Class']

## Create Simulated Datasets

In [ ]:
X_simulated_small, y_simulated_small = make_classification(    n_samples=300,    n_features=6,    n_classes=2,    random_state=1)X_simulated_large, y_simulated_large = make_classification(    n_samples=15000,    n_features=6,    n_classes=2,    random_state=1)

## Exploratory Data Analysis

In [ ]:
flights_per_airline = df.groupby('Airline').agg({'Class':'count'})delays_by_airline = df.groupby('Airline').agg({'Class':'sum'})flights_per_airline['%_delays'] = 100*delays_by_airline['Class']/flights_per_airline['Class']

In [ ]:
plt.figure(figsize=(10,7))sns.countplot(y='Airline', hue='Class', data=df)plt.title('Number of Flights On Time and Delayed by Airline')plt.show()

In [ ]:
sns.barplot(y='Airline', x='%_delays', data=flights_per_airline, orient='h', color='C0')plt.xlabel('Percent of Flights Delayed')plt.ylabel('Airline')plt.show()

In [ ]:
delayed = df[df['Class']==1]plt.figure(figsize=(10,5))plt.hist([df['Length'], delayed['Length']], label=['All','Delayed'], density=True)plt.legend()plt.xlabel('Flight Length')plt.show()

In [ ]:
plt.figure(figsize=(10,5))plt.hist([df['DayOfWeek'], delayed['DayOfWeek']], label=['All','Delayed'], density=True)plt.legend()plt.xlabel('Day Of Week')plt.show()

In [ ]:
time_bins = [0]initial_time = 0for i in range(24):    initial_time += 60    time_bins.append(initial_time)df['TimeBin'] = pd.cut(df['Time'], bins=time_bins, include_lowest=True, right=False)delays = df[df['Class']==1]delays['TimeBin'] = pd.cut(delays['Time'], bins=time_bins, include_lowest=True, right=False)counts = delays.groupby(['DayOfWeek','TimeBin']).size().unstack(fill_value=0)plt.figure(figsize=(12,8))sns.heatmap(counts, cmap='viridis')plt.title('Delays by Day of Week and Time of Day')plt.show()